# Table S62 - Variability and estimation errors in parameters estimation

Reads `.Rds` simulation outputs from the **enhanced** pipeline and produces
the LaTeX table (Accuracy + ARI, by censoring / k / γ, scenario = `baseline`).

*THE FOLLOWING DATA ARE USED FOR THIS SIMULATION:*

**Expected directory layout** (mirrors `enhanced_simulation_main.R`):
```
output/
  tab2/
    baseline/
      sim_seed*_c*_k*_gammapar*_frailty*_censor*.Rds
```

In [45]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata      
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR  = Path("output/tab2")   # root output folder from enhanced_simulation_main.R
SCENARIO  = "baseline"            # which sub-folder / scenario to analyse
N_CLUSTERS_EXPECTED = 3           # keep only runs where n_components == 3

folder = BASE_DIR / SCENARIO
rds_files = sorted(folder.glob("*.Rds"))
print(f"Found {len(rds_files)} .Rds files in '{folder}'")

Found 2800 .Rds files in 'output/tab2/baseline'


In [46]:
# ── Load & process all .Rds files ─────────────────────────────────────────

records = []
skipped = 0

for file in rds_files:
    fname = file.name

    # ── Determine censoring from filename ──────────────────────────────
    if "censoradministrative" in fname:
        censor_name = "Administrative"
    elif "censornormal" in fname:
        censor_name = "Normal"
    else:
        skipped += 1
        continue

    # ── Read the Rds file ──────────────────────────────────────────────
    try:
        obj = rdata.read_rds(file)
        obj = {str(k): v for k, v in obj.items()}

    except Exception as e:
        print(f"  [WARNING] Could not read {fname}: {e}")
        skipped += 1
        continue

    record = {
        "seed": float(obj["seed"][0]),
        "c": float(obj["c"][0]),
        "k": float(obj["k"][0]),
        "gammapar": float(obj["gammapar"][0]),
        "n_components": float(obj["n_components"][0]),
        "loglik": float(obj["loglik"][0]),
        "iterations": float(obj["iterations"][0]),
        "lambdaoptim": float(obj["lambdaoptim"][0]),
        "censor": censor_name
    }

    # expand the estimates into individual columns
    for i, val in enumerate(obj["estimate"]):
        record[f"estimate_{i}"] = float(val)

    records.append(record)

print(f"Processed {len(rds_files) - skipped} files  |  skipped {skipped}")

# build dataframe
df_pen = pd.DataFrame(records)

### now we transform back the estimate_2 which is $\xi^\rho$ to make \xi
df_pen['estimate_2'] = df_pen['estimate_2']**(1/df_pen['estimate_1'])

df_pen


/opt/homebrew/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "table". The underlying R object is returned instead.
  warnings.warn(


Processed 2800 files  |  skipped 0


,seed,c,k,gammapar,n_components,loglik,iterations,lambdaoptim,censor,estimate_0,estimate_1,estimate_2,estimate_3
0,0.0,3.0,20.0,0.001000,3.0,-795.372050,3.0,1000.000000,Administrative,1.050744,2.509115,0.009995,0.698357
1,0.0,3.0,20.0,0.001000,3.0,-938.763520,36.0,484.580744,Normal,1.012431,2.441915,0.010139,0.675525
2,0.0,3.0,20.0,0.010000,3.0,-807.438995,29.0,79.523954,Administrative,0.985668,2.362428,0.010125,0.650767
3,0.0,3.0,20.0,0.010000,3.0,-960.498793,35.0,343.002153,Normal,0.873973,2.221563,0.010338,0.603733
4,0.0,3.0,20.0,0.100000,3.0,-878.503257,34.0,117.925534,Administrative,0.750998,1.737520,0.011037,0.438235
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,9.0,3.0,50.0,0.400000,3.0,-1551.972208,7.0,13.530112,Normal,0.062153,0.515234,0.006816,0.008611
2796,9.0,3.0,50.0,0.000100,3.0,-818.512464,3.0,1000.000000,Administrative,0.612546,2.547487,0.009654,0.716774
2797,9.0,3.0,50.0,0.000100,3.0,-927.052041,3.0,1440.000000,Normal,0.620286,2.577957,0.009797,0.722053
2798,9.0,3.0,50.0,0.000001,3.0,-818.104933,3.0,1000.000000,Administrative,0.613633,2.552470,0.009647,0.718366


In [47]:
df_pen[df_pen.k == 20].groupby(['k'])['iterations'].value_counts()

k     iterations
20.0  3.0           404
      36.0           57
      35.0           54
      33.0           47
      38.0           47
                   ... 
      61.0            1
      50.0            1
      57.0            1
      55.0            1
      12.0            1
Name: count, Length: 61, dtype: int64

In [48]:
df_pen[df_pen.k == 50].groupby(['k'])['iterations'].value_counts()

k     iterations
50.0  3.0           303
      28.0          100
      29.0           82
      27.0           73
      26.0           65
      24.0           60
      21.0           57
      23.0           51
      25.0           50
      30.0           43
      20.0           43
      8.0            40
      22.0           40
      31.0           38
      19.0           32
      18.0           28
      7.0            26
      17.0           24
      16.0           23
      10.0           22
      200.0          22
      6.0            17
      13.0           17
      14.0           16
      9.0            16
      12.0           16
      33.0           15
      15.0           15
      11.0           15
      32.0           14
      4.0            10
      5.0             6
      34.0            6
      37.0            4
      36.0            4
      35.0            3
      54.0            1
      60.0            1
      42.0            1
      53.0            1
Name: count, dtype: int

# for the first analysis

In [49]:
import pandas as pd

summary_iter = (
    df_pen
    .groupby(["k", "gammapar"])
    .apply(lambda x: pd.Series({
        "pct_replic_3clusters": (x["n_components"] == 3).mean() * 100,
        "avg_iterations_if_3clusters": x.loc[x["n_components"] == 3, "iterations"].mean()
    }))
    .reset_index()
)

print(summary_iter)


       k  gammapar  pct_replic_3clusters  avg_iterations_if_3clusters
0   20.0  0.000001                 100.0                    13.070000
1   20.0  0.000100                 100.0                    12.810000
2   20.0  0.001000                 100.0                    18.550000
3   20.0  0.010000                  99.5                    27.095477
4   20.0  0.100000                  99.0                    32.722222
5   20.0  0.200000                  99.5                    28.703518
6   20.0  0.400000                  97.0                    24.603093
7   50.0  0.000001                 100.0                    12.040000
8   50.0  0.000100                 100.0                    11.540000
9   50.0  0.001000                 100.0                    21.700000
10  50.0  0.010000                 100.0                    23.935000
11  50.0  0.100000                  99.0                    29.121212
12  50.0  0.200000                  99.5                    21.557789
13  50.0  0.400000  

In [50]:
import pandas as pd

summary_iter = (
    df_pen
    .groupby(["censor"])
    .apply(lambda x: pd.Series({
        "avg_iterations_if_3clusters": x.loc[x["n_components"] == 3, "iterations"].mean()
    }))
    .reset_index()
)

print(summary_iter)


           censor  avg_iterations_if_3clusters
0  Administrative                    17.490267
1          Normal                    24.483477


In [51]:
import pandas as pd

summary_iter = (
    df_pen
    .groupby(["censor", "gammapar"])
    .apply(lambda x: pd.Series({
        "pct_replic_3clusters": (x["n_components"] == 3).mean() * 100,
        "avg_iterations_if_3clusters": x.loc[x["n_components"] == 3, "iterations"].mean()
    }))
    .reset_index()
)

print(summary_iter)


            censor  gammapar  pct_replic_3clusters  \
0   Administrative  0.000001                 100.0   
1   Administrative  0.000100                 100.0   
2   Administrative  0.001000                 100.0   
3   Administrative  0.010000                 100.0   
4   Administrative  0.100000                  99.5   
5   Administrative  0.200000                  99.0   
6   Administrative  0.400000                  95.0   
7           Normal  0.000001                 100.0   
8           Normal  0.000100                 100.0   
9           Normal  0.001000                 100.0   
10          Normal  0.010000                  99.5   
11          Normal  0.100000                  98.5   
12          Normal  0.200000                 100.0   
13          Normal  0.400000                  98.0   

    avg_iterations_if_3clusters  
0                      7.975000  
1                      8.005000  
2                     13.540000  
3                     22.865000  
4                  

# now we import the non-penalized ones

In [52]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata      
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR  = Path("output/tab2")   # root output folder from enhanced_simulation_main.R
SCENARIO  = "baseline/notpenalized"    # which sub-folder / scenario to analyse
N_CLUSTERS_EXPECTED = 3           # keep only runs where n_components == 3

folder = BASE_DIR / SCENARIO
rds_files = sorted(folder.glob("*.Rds"))
print(f"Found {len(rds_files)} .Rds files in '{folder}'")

Found 400 .Rds files in 'output/tab2/baseline/notpenalized'


In [53]:
# ── Load & process all .Rds files ─────────────────────────────────────────

records = []
skipped = 0

for file in rds_files:
    fname = file.name

    # ── Determine censoring from filename ──────────────────────────────
    if "censoradministrative" in fname:
        censor_name = "Administrative"
    elif "censornormal" in fname:
        censor_name = "Normal"
    else:
        skipped += 1
        continue

    # ── Read the Rds file ──────────────────────────────────────────────
    try:
        obj = rdata.read_rds(file)
        obj = {str(k): v for k, v in obj.items()}

    except Exception as e:
        print(f"  [WARNING] Could not read {fname}: {e}")
        skipped += 1
        continue

    record = {
        "seed": float(obj["seed"][0]),
        "c": float(obj["c"][0]),
        "k": float(obj["k"][0]),
        "gammapar": float(obj["gammapar"][0]),
        "loglik": float(obj["loglik"][0]),
        "iterations": float(obj["iterations"][0]),
        "lambdaoptim": float(obj["lambdaoptim"][0]),
        "censor": censor_name
    }

    # expand the estimates into individual columns
    for i, val in enumerate(obj["estimate"]):
        record[f"estimate_{i}"] = float(val)

    records.append(record)

print(f"Processed {len(rds_files) - skipped} files  |  skipped {skipped}")

# build dataframe
df_notpen = pd.DataFrame(records)

### now we transform back the estimate_2 which is $\xi^\rho$ to make \xi
df_notpen['estimate_2'] = df_notpen['estimate_2']**(1/df_notpen['estimate_1'])

df_notpen


Processed 400 files  |  skipped 0


,seed,c,k,gammapar,loglik,iterations,lambdaoptim,censor,estimate_0,estimate_1,estimate_2,estimate_3
0,0.0,3.0,20.0,0.0,-793.788792,0.0,1000.0,Administrative,1.059279,2.529220,0.009977,0.704828
1,0.0,3.0,20.0,0.0,-935.433255,0.0,1000.0,Normal,1.037181,2.479198,0.010106,0.687399
2,0.0,3.0,50.0,0.0,-793.788792,0.0,1000.0,Administrative,1.059279,2.529220,0.009977,0.704828
3,0.0,3.0,50.0,0.0,-935.433255,0.0,1000.0,Normal,1.037181,2.479198,0.010106,0.687399
4,10.0,3.0,20.0,0.0,-886.795026,0.0,1000.0,Administrative,0.614411,2.462905,0.009931,0.683281
...,...,...,...,...,...,...,...,...,...,...,...,...
395,99.0,3.0,50.0,0.0,-1036.916620,0.0,1000.0,Normal,0.835105,2.546825,0.009964,0.701098
396,9.0,3.0,20.0,0.0,-818.100811,0.0,1000.0,Administrative,0.613617,2.552513,0.009643,0.718344
397,9.0,3.0,20.0,0.0,-926.159745,0.0,1000.0,Normal,0.621027,2.588661,0.009789,0.725276
398,9.0,3.0,50.0,0.0,-818.100811,0.0,1000.0,Administrative,0.613617,2.552513,0.009643,0.718344


# Now we concatenate the two dataframes

In [54]:
df = pd.concat([df_notpen, df_pen], axis = 0)
df

,seed,c,k,gammapar,loglik,iterations,lambdaoptim,censor,estimate_0,estimate_1,estimate_2,estimate_3,n_components
0,0.0,3.0,20.0,0.000000,-793.788792,0.0,1000.000000,Administrative,1.059279,2.529220,0.009977,0.704828,NaN
1,0.0,3.0,20.0,0.000000,-935.433255,0.0,1000.000000,Normal,1.037181,2.479198,0.010106,0.687399,NaN
2,0.0,3.0,50.0,0.000000,-793.788792,0.0,1000.000000,Administrative,1.059279,2.529220,0.009977,0.704828,NaN
3,0.0,3.0,50.0,0.000000,-935.433255,0.0,1000.000000,Normal,1.037181,2.479198,0.010106,0.687399,NaN
4,10.0,3.0,20.0,0.000000,-886.795026,0.0,1000.000000,Administrative,0.614411,2.462905,0.009931,0.683281,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,9.0,3.0,50.0,0.400000,-1551.972208,7.0,13.530112,Normal,0.062153,0.515234,0.006816,0.008611,3.0
2796,9.0,3.0,50.0,0.000100,-818.512464,3.0,1000.000000,Administrative,0.612546,2.547487,0.009654,0.716774,3.0
2797,9.0,3.0,50.0,0.000100,-927.052041,3.0,1440.000000,Normal,0.620286,2.577957,0.009797,0.722053,3.0
2798,9.0,3.0,50.0,0.000001,-818.104933,3.0,1000.000000,Administrative,0.613633,2.552470,0.009647,0.718366,3.0


In [55]:
df.n_components.value_counts()

n_components
3.0    2779
2.0      19
1.0       1
4.0       1
Name: count, dtype: int64

In [56]:
df.loc[df.gammapar!=0, 'n_components'].value_counts()

n_components
3.0    2779
2.0      19
1.0       1
4.0       1
Name: count, dtype: int64

In [57]:
df = df.loc[(df['n_components'] == 3) | (df['n_components'].isna())]

In [58]:
df = df.loc[(df['k'] == 20)] # ANALYSIS RESTRICTED TO K=20

In [59]:
df

,seed,c,k,gammapar,loglik,iterations,lambdaoptim,censor,estimate_0,estimate_1,estimate_2,estimate_3,n_components
0,0.0,3.0,20.0,0.000000,-793.788792,0.0,1000.000000,Administrative,1.059279,2.529220,0.009977,0.704828,NaN
1,0.0,3.0,20.0,0.000000,-935.433255,0.0,1000.000000,Normal,1.037181,2.479198,0.010106,0.687399,NaN
4,10.0,3.0,20.0,0.000000,-886.795026,0.0,1000.000000,Administrative,0.614411,2.462905,0.009931,0.683281,NaN
5,10.0,3.0,20.0,0.000000,-986.926188,0.0,1000.000000,Normal,0.632612,2.463794,0.009926,0.684959,NaN
8,11.0,3.0,20.0,0.000000,-842.560518,0.0,1000.000000,Administrative,0.714209,2.462628,0.010128,0.693482,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2781,9.0,3.0,20.0,0.400000,-1130.039126,25.0,97.404321,Normal,0.659016,1.327022,0.016189,0.298467,3.0
2782,9.0,3.0,20.0,0.000100,-818.237810,3.0,1000.000000,Administrative,0.613309,2.551160,0.009648,0.717902,3.0
2783,9.0,3.0,20.0,0.000100,-926.522725,3.0,1440.000000,Normal,0.620930,2.584001,0.009796,0.723861,3.0
2784,9.0,3.0,20.0,0.000001,-818.102252,3.0,1000.000000,Administrative,0.613617,2.552513,0.009643,0.718344,3.0


# Bias of Estimators

In [ ]:
# Filter & convert
dfx = df.loc[df.censor == 'Administrative', ['loglik', 'gammapar']].apply(pd.to_numeric, errors='coerce')

In [61]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

# True parameter values
true_values = {
    'estimate_0': 0.5,
    'estimate_1': 2.5,
    'estimate_2': 0.01,
    'estimate_3': np.log(2)
}

# Titles
titles = {
    'estimate_0': r"(a) $\theta$",
    'estimate_1': r"(b) $\rho$",
    'estimate_2': r"(c) $\xi$",
    'estimate_3': r"(d) $\beta$"
}

censoring = {
    'Administrative': "(i) Administrative",
    'Normal': "(ii) Normal"
}
    

TITLE_SIZE = 16
LABEL_SIZE = 14
TICK_SIZE  = 10

estimates = list(titles.keys())

for j in ['Administrative', 'Normal']:
    
    fig, axes = plt.subplots(
        1, len(estimates),
        figsize=(4.2 * len(estimates), 4.7),
        sharey=False
    )

    # UNIQUE gamma values for this censoring type
    gamma_vals = sorted(df.loc[df.censor == j, "gammapar"].dropna().unique())
    palette = sns.color_palette("Greys", len(gamma_vals))

    for ax, est in zip(axes, estimates):

        # Filter
        dfx = df.loc[df.censor == j, [est, 'gammapar']].apply(pd.to_numeric, errors='coerce')

        # Boxplot
        sns.boxplot(
            data=dfx,
            x="gammapar",
            y=est,
            hue="gammapar",
            palette=palette,
            dodge=False,
            ax=ax
        )

        # True horizontal line
        ax.axhline(
            y=true_values[est],
            color='red',
            linestyle='--'
        )

        # X-axis with gamma values
        ax.set_xlabel(r"$\gamma$", fontsize=LABEL_SIZE)

        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

        ax.tick_params(axis='x', labelsize=TICK_SIZE)
        ax.tick_params(axis='y', labelsize=TICK_SIZE)

        ax.grid(axis='both', linestyle='--', alpha=0.5)

        # Title
        ax.set_ylabel("")
        ax.set_title(f"{titles[est]}", fontsize=TITLE_SIZE)

        # Remove subplot legend
        ax.get_legend().remove()

    # Build global legend manually
    # handles = [plt.Line2D([0], [0], color=palette[i], lw=8)
    #            for i in range(len(gamma_vals))]
    # labels = [f"{g}" for g in gamma_vals]

    # fig.subplots_adjust(bottom=0.3, top=0.8)

    # fig.legend(
    #     handles, labels,
    #     title=r"$\gamma$",
    #     loc="lower center",
    #     ncol=len(labels),
    #     fontsize=TICK_SIZE,
    #     title_fontsize=TICK_SIZE
    # )

    # Global title
    plt.suptitle(
        rf"Parameter estimates – {censoring[j]} censoring",
        fontsize=TITLE_SIZE
    )

    fig.subplots_adjust(top=0.85, bottom=0.3)

    plt.savefig(f"Plots/Estimates_{j}.pdf", bbox_inches="tight")
    plt.close()


/var/folders/lg/0lc2g3rx12jf4ztglsz2bgc80000gn/T/ipykernel_65306/3517988644.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
/var/folders/lg/0lc2g3rx12jf4ztglsz2bgc80000gn/T/ipykernel_65306/3517988644.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
/var/folders/lg/0lc2g3rx12jf4ztglsz2bgc80000gn/T/ipykernel_65306/3517988644.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
/var/folders/lg/0lc2g3rx12jf4ztglsz2bgc80000gn/T/ipykernel_65306/3517988644.py:74: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, 

# Bias-variance trade-off

In [62]:
import pandas as pd
import numpy as np

# true values
truevals = {
    'estimate_0': 0.5,
    'estimate_1': 2.5,
    'estimate_2': 0.01,
    'estimate_3': np.log(2)
}

# nice names for columns
rename_map = {
    'estimate_0': 'theta',
    'estimate_1': 'rho',
    'estimate_2': 'xi',
    'estimate_3': 'beta'
}

rows = []

for censor in ['Administrative', 'Normal']:
    dfx = df[df.censor == censor].copy()

    for gamma in sorted(dfx.gammapar.unique()):
        df_sub = dfx[dfx.gammapar == gamma]

        # --- MSE ---
        mse_vals = {}
        for col, trueval in truevals.items():
            mse_vals[rename_map[col]] = np.mean((df_sub[col] - trueval)**2)

        # --- Variance ---
        var_vals = {}
        for col in truevals.keys():
            vals = df_sub[col]
            var_vals[rename_map[col]] = np.mean((vals - vals.mean())**2)

        # store row
        rows.append({
            "Censoring": censor,
            "gamma": gamma,
            "MSE_theta": mse_vals["theta"],
            "Var_theta": var_vals["theta"],
            "MSE_rho": mse_vals["rho"],
            "Var_rho": var_vals["rho"],
            "MSE_xi": mse_vals["xi"],
            "Var_xi": var_vals["xi"],
            "MSE_beta": mse_vals["beta"],
            "Var_beta": var_vals["beta"]
        })

# final table
table_df = pd.DataFrame(rows)

# sort nicely
table_df = table_df.sort_values(by=["Censoring", "gamma"]) #.round(3)


In [63]:
table_df

,Censoring,gamma,MSE_theta,Var_theta,MSE_rho,Var_rho,MSE_xi,Var_xi,MSE_beta,Var_beta
0,Administrative,0.000000,0.052496,0.051343,0.012922,0.012844,8.796648e-07,8.789405e-07,0.000935,0.000929
1,Administrative,0.000001,0.052490,0.051335,0.012926,0.012848,8.794181e-07,8.786979e-07,0.000935,0.000929
2,Administrative,0.000100,0.052462,0.051305,0.012911,0.012806,8.806351e-07,8.797939e-07,0.000935,0.000926
3,Administrative,0.001000,0.051660,0.050390,0.013026,0.012499,8.842367e-07,8.816798e-07,0.000958,0.000907
4,Administrative,0.010000,0.048617,0.046479,0.030196,0.011468,9.930679e-07,9.222641e-07,0.002887,0.000905
5,Administrative,0.100000,0.069151,0.069131,0.506408,0.024815,6.830422e-06,2.249595e-06,0.057731,0.002857
6,Administrative,0.200000,0.095157,0.090875,0.934312,0.026553,1.633226e-05,3.825475e-06,0.109356,0.003274
7,Administrative,0.400000,0.116032,0.096265,1.525272,0.034237,3.109878e-05,5.684108e-06,0.182010,0.004142
8,Normal,0.000000,0.053610,0.052442,0.012142,0.012090,7.858351e-07,7.839582e-07,0.000932,0.000926
9,Normal,0.000001,0.053612,0.052439,0.012144,0.012092,7.866239e-07,7.847435e-07,0.000932,0.000926


In [64]:
import pandas as pd

table_df["NMSE_theta"] = table_df["MSE_theta"] / table_df["Var_theta"]
table_df["NMSE_rho"]   = table_df["MSE_rho"]   / table_df["Var_rho"]
table_df["NMSE_xi"]    = table_df["MSE_xi"]    / table_df["Var_xi"]
table_df["NMSE_beta"]  = table_df["MSE_beta"]  / table_df["Var_beta"]

# Round if desired
table_df = table_df.round(3)

table_df

,Censoring,gamma,MSE_theta,Var_theta,MSE_rho,Var_rho,MSE_xi,Var_xi,MSE_beta,Var_beta,NMSE_theta,NMSE_rho,NMSE_xi,NMSE_beta
0,Administrative,0.000,0.052,0.051,0.013,0.013,0.0,0.0,0.001,0.001,1.022,1.006,1.001,1.007
1,Administrative,0.000,0.052,0.051,0.013,0.013,0.0,0.0,0.001,0.001,1.022,1.006,1.001,1.007
2,Administrative,0.000,0.052,0.051,0.013,0.013,0.0,0.0,0.001,0.001,1.023,1.008,1.001,1.010
3,Administrative,0.001,0.052,0.050,0.013,0.012,0.0,0.0,0.001,0.001,1.025,1.042,1.003,1.056
4,Administrative,0.010,0.049,0.046,0.030,0.011,0.0,0.0,0.003,0.001,1.046,2.633,1.077,3.190
5,Administrative,0.100,0.069,0.069,0.506,0.025,0.0,0.0,0.058,0.003,1.000,20.407,3.036,20.207
6,Administrative,0.200,0.095,0.091,0.934,0.027,0.0,0.0,0.109,0.003,1.047,35.186,4.269,33.404
7,Administrative,0.400,0.116,0.096,1.525,0.034,0.0,0.0,0.182,0.004,1.205,44.551,5.471,43.947
8,Normal,0.000,0.054,0.052,0.012,0.012,0.0,0.0,0.001,0.001,1.022,1.004,1.002,1.007
9,Normal,0.000,0.054,0.052,0.012,0.012,0.0,0.0,0.001,0.001,1.022,1.004,1.002,1.007


In [65]:
def latex_row(values):
    return " & ".join([f"{v:.3f}" if isinstance(v, float) else str(v) for v in values]) + r" \\"

for censor in ["Administrative", "Normal"]:
    dfc = table_df[table_df.Censoring == censor]
    for _, row in dfc.iterrows():
        print(
            latex_row([
                row["Censoring"],
                row["gamma"],
                row["NMSE_theta"], row["Var_theta"],
                row["NMSE_rho"], row["Var_rho"],
                row["NMSE_xi"], row["Var_xi"],
                row["NMSE_beta"], row["Var_beta"]
            ])
        )

Administrative & 0.000 & 1.022 & 0.051 & 1.006 & 0.013 & 1.001 & 0.000 & 1.007 & 0.001 \\
Administrative & 0.000 & 1.022 & 0.051 & 1.006 & 0.013 & 1.001 & 0.000 & 1.007 & 0.001 \\
Administrative & 0.000 & 1.023 & 0.051 & 1.008 & 0.013 & 1.001 & 0.000 & 1.010 & 0.001 \\
Administrative & 0.001 & 1.025 & 0.050 & 1.042 & 0.012 & 1.003 & 0.000 & 1.056 & 0.001 \\
Administrative & 0.010 & 1.046 & 0.046 & 2.633 & 0.011 & 1.077 & 0.000 & 3.190 & 0.001 \\
Administrative & 0.100 & 1.000 & 0.069 & 20.407 & 0.025 & 3.036 & 0.000 & 20.207 & 0.003 \\
Administrative & 0.200 & 1.047 & 0.091 & 35.186 & 0.027 & 4.269 & 0.000 & 33.404 & 0.003 \\
Administrative & 0.400 & 1.205 & 0.096 & 44.551 & 0.034 & 5.471 & 0.000 & 43.947 & 0.004 \\
Normal & 0.000 & 1.022 & 0.052 & 1.004 & 0.012 & 1.002 & 0.000 & 1.007 & 0.001 \\
Normal & 0.000 & 1.022 & 0.052 & 1.004 & 0.012 & 1.002 & 0.000 & 1.007 & 0.001 \\
Normal & 0.000 & 1.024 & 0.052 & 1.009 & 0.012 & 1.003 & 0.000 & 1.013 & 0.001 \\
Normal & 0.001 & 1.036 & 0.0